# Mentor Learning Analytics Example

This notebook demonstrates how to analyze exported learning trajectory data from Mentor.

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from research.export.trajectory_exporter import AnonymizedDataset

## Load Exported Data

First, load an exported dataset. Replace the path with your export directory.

In [ ]:
# Load exported dataset
export_path = Path("./exports/course_example")
dataset = AnonymizedDataset(export_path)

# Load data
interactions = dataset.load_interactions()
mastery = dataset.load_mastery()
summary = dataset.load_summary()

print(f"Total students: {summary['total_students']}")
print(f"Total interactions: {summary['total_interactions']}")

## Convert to DataFrames

In [ ]:
# Convert to pandas DataFrames for analysis
df_interactions = pd.DataFrame(interactions)
df_interactions['timestamp'] = pd.to_datetime(df_interactions['timestamp'])

df_mastery = pd.DataFrame(mastery)

df_interactions.head()

## Analyze Pedagogical Move Distribution

In [ ]:
# Pedagogical move distribution
move_counts = df_interactions['pedagogical_move'].value_counts()

plt.figure(figsize=(10, 6))
move_counts.plot(kind='bar')
plt.title('Distribution of Pedagogical Moves')
plt.xlabel('Pedagogical Move')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Learning Curves Analysis

In [ ]:
# Compute learning curves
learning_curves = dataset.compute_learning_curves()

plt.figure(figsize=(12, 6))

# Plot individual student curves with low alpha
for student_id, curve in learning_curves.items():
    if len(curve) > 5:  # Only plot students with enough data
        x, y = zip(*curve)
        plt.plot(x, y, alpha=0.3, linewidth=1)

# Compute and plot average curve
max_len = max(len(curve) for curve in learning_curves.values())
avg_curve = []
for i in range(max_len):
    values = [curve[i][1] for curve in learning_curves.values() if len(curve) > i]
    if values:
        avg_curve.append(sum(values) / len(values))

plt.plot(range(len(avg_curve)), avg_curve, 'k-', linewidth=3, label='Average')

plt.xlabel('Interaction Number')
plt.ylabel('Mastery Level')
plt.title('Learning Curves Across Students')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Concept Difficulty Analysis

In [ ]:
# Analyze concept difficulty by average mastery
concept_mastery = df_mastery.groupby('concept_name')['mastery_level'].agg(['mean', 'std', 'count'])
concept_mastery = concept_mastery.sort_values('mean')

plt.figure(figsize=(12, 6))
plt.barh(range(len(concept_mastery)), concept_mastery['mean'], xerr=concept_mastery['std'])
plt.yticks(range(len(concept_mastery)), concept_mastery.index)
plt.xlabel('Average Mastery Level')
plt.title('Concept Difficulty (Lower = Harder)')
plt.tight_layout()
plt.show()

## Time-on-Task Analysis

In [ ]:
# Analyze response times
df_interactions['response_time_ms'] = pd.to_numeric(df_interactions['response_time_ms'], errors='coerce')

# Filter outliers
response_times = df_interactions['response_time_ms'].dropna()
response_times = response_times[response_times < response_times.quantile(0.95)]

plt.figure(figsize=(10, 6))
plt.hist(response_times / 1000, bins=50, edgecolor='black')
plt.xlabel('Response Time (seconds)')
plt.ylabel('Frequency')
plt.title('Distribution of Student Response Times')
plt.tight_layout()
plt.show()

print(f"Median response time: {response_times.median() / 1000:.1f} seconds")
print(f"Mean response time: {response_times.mean() / 1000:.1f} seconds")

## Mastery Change by Pedagogical Move

In [ ]:
# Calculate mastery change per interaction
df_interactions['mastery_change'] = (
    pd.to_numeric(df_interactions['mastery_after'], errors='coerce') - 
    pd.to_numeric(df_interactions['mastery_before'], errors='coerce')
)

# Average mastery change by pedagogical move
move_effectiveness = df_interactions.groupby('pedagogical_move')['mastery_change'].agg(['mean', 'std', 'count'])
move_effectiveness = move_effectiveness[move_effectiveness['count'] >= 10]  # Filter low counts
move_effectiveness = move_effectiveness.sort_values('mean', ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in move_effectiveness['mean']]
plt.barh(range(len(move_effectiveness)), move_effectiveness['mean'], color=colors)
plt.yticks(range(len(move_effectiveness)), move_effectiveness.index)
plt.xlabel('Average Mastery Change')
plt.title('Effectiveness of Pedagogical Moves')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## Session-Level Analysis

In [ ]:
# Analyze sessions
session_stats = df_interactions.groupby(['student_id', 'session_id']).agg({
    'turn_number': 'max',
    'mastery_change': 'sum',
    'timestamp': ['min', 'max']
}).reset_index()

session_stats.columns = ['student_id', 'session_id', 'turns', 'total_mastery_change', 'start_time', 'end_time']
session_stats['duration_minutes'] = (session_stats['end_time'] - session_stats['start_time']).dt.total_seconds() / 60

# Session length distribution
plt.figure(figsize=(10, 6))
plt.hist(session_stats['turns'], bins=30, edgecolor='black')
plt.xlabel('Number of Turns')
plt.ylabel('Frequency')
plt.title('Distribution of Session Lengths')
plt.tight_layout()
plt.show()

print(f"Average turns per session: {session_stats['turns'].mean():.1f}")
print(f"Average session duration: {session_stats['duration_minutes'].mean():.1f} minutes")